# 구조 확인

In [139]:
import re
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


# driver
def create_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1200")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


# 유틸
def normalize_url(url: str) -> str:
    parsed = urlparse(url)
    return parsed._replace(fragment="").geturl()


def is_same_domain(url: str, base_url: str) -> bool:
    return urlparse(url).netloc == urlparse(base_url).netloc


def common_noise_patterns():
    return [
        "로그인", "Login", "사이트맵", "SITEMAP", "English", "KUPID",
        "HY-in", "전체메뉴", "본문 바로가기", "주메뉴 바로가기",
        "서브메뉴 바로가기", "푸터 바로가기", "내용으로 건너뛰기",
        "Korean", "포털", "검색"
    ]


def is_dead_page(title: str, text: str) -> bool:
    combined = f"{title}\n{text}".lower()
    dead_patterns = [
        "요청하신 페이지가 존재하지 않습니다",
        "존재하지 않습니다",
        "페이지를 찾을 수 없습니다",
        "404",
        "not found",
    ]
    return any(p.lower() in combined for p in dead_patterns)


# 페이지 로드
def load_page(driver, url: str):
    driver.get(url)
    time.sleep(2)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    full_text = soup.get_text("\n", strip=True)
    return title, full_text, soup


# 본문만 추출
def extract_clean_main_text(driver):
    selectors = [
        "main", "#content", ".contents", ".sub_content",
        ".content", ".cont", ".board_view", ".view",
        ".txt", ".con_box", "article"
    ]

    noise_patterns = common_noise_patterns()

    for sel in selectors:
        try:
            elems = driver.find_elements("css selector", sel)
            if not elems:
                continue

            text = elems[0].text.strip()
            if len(text) < 100:
                continue

            cleaned = text
            for noise in noise_patterns:
                cleaned = cleaned.replace(noise, "")

            cleaned = re.sub(r"\n{2,}", "\n", cleaned).strip()

            if len(cleaned) >= 100:
                return {
                    "selector": sel,
                    "raw_text_len": len(text),
                    "clean_text_len": len(cleaned),
                    "text": cleaned
                }

        except Exception:
            continue

    return {
        "selector": None,
        "raw_text_len": 0,
        "clean_text_len": 0,
        "text": ""
    }


def inspect_main_candidates(driver):
    selectors = [
        "main", "#content", ".content", ".contents",
        ".sub_content", ".cont", ".board_view",
        ".view", ".txt", ".con_box", "article", "body"
    ]

    rows = []
    for sel in selectors:
        try:
            elems = driver.find_elements("css selector", sel)
            if not elems:
                continue
            text = elems[0].text.strip()
            rows.append({
                "selector": sel,
                "text_len": len(text),
                "sample": text[:200].replace("\n", " ")
            })
        except Exception:
            continue

    return pd.DataFrame(rows)

In [140]:
# 상위 홈페이지 구조 확인
def inspect_parent_site(driver, site_name: str, url: str, dept_keywords: list[str], max_links: int = 120):
    title, full_text, soup = load_page(driver, url)
    clean_main = extract_clean_main_text(driver)
    selector_df = inspect_main_candidates(driver)

    links = []
    for a in soup.find_all("a", href=True):
        href = normalize_url(urljoin(url, a["href"]))
        link_text = a.get_text(" ", strip=True)

        if not link_text:
            continue
        if not is_same_domain(href, url):
            continue

        matched_dept = None
        for kw in dept_keywords:
            if kw in link_text or kw in href:
                matched_dept = kw
                break

        links.append({
            "site_name": site_name,
            "dept_match": matched_dept,
            "link_text": link_text,
            "href": href
        })

    df_links = pd.DataFrame(links).drop_duplicates()

    print("=" * 120)
    print("[상위 홈페이지 구조 확인]")
    print("SITE NAME :", site_name)
    print("URL       :", url)
    print("TITLE     :", title)
    print("DEAD PAGE :", is_dead_page(title, full_text))

    print("\n[본문 selector 후보]")
    print(selector_df if not selector_df.empty else "후보 없음")

    print("\n[본문만 추출 결과]")
    print("selector      :", clean_main["selector"])
    print("raw_text_len  :", clean_main["raw_text_len"])
    print("clean_text_len:", clean_main["clean_text_len"])
    print("sample        :", clean_main["text"][:400])

    print("\n[학과/전공 키워드 매칭 링크]")
    matched = df_links[df_links["dept_match"].notna()] if not df_links.empty else pd.DataFrame()
    print(matched.head(max_links) if not matched.empty else "매칭 링크 없음")
    print("=" * 120)

    return {
        "title": title,
        "full_text": full_text,
        "clean_main": clean_main,
        "selector_df": selector_df,
        "links_df": df_links
    }


# 상위 홈페이지 내부 학과/전공 페이지 구조 확인
def inspect_child_pages_from_parent(driver, parent_url: str, dept_keywords: list[str], max_child_pages: int = 10):
    """
    상위 홈페이지 내부에서 학과/전공 키워드가 걸린 링크를 추려서
    그 하위 페이지 구조까지 확인
    """
    _, _, soup = load_page(driver, parent_url)

    child_links = []
    for a in soup.find_all("a", href=True):
        href = normalize_url(urljoin(parent_url, a["href"]))
        link_text = a.get_text(" ", strip=True)

        if not link_text:
            continue
        if not is_same_domain(href, parent_url):
            continue

        matched = None
        for kw in dept_keywords:
            if kw in link_text or kw in href:
                matched = kw
                break

        if matched:
            child_links.append({
                "dept_match": matched,
                "link_text": link_text,
                "href": href
            })

    df_child = pd.DataFrame(child_links).drop_duplicates()

    if df_child.empty:
        print("\n[하위 학과/전공 링크 없음]")
        return {}

    results = {}
    for _, row in df_child.head(max_child_pages).iterrows():
        site_name = f"{row['dept_match']} | {row['link_text']}"
        print(f"\n\n##### 하위 페이지 확인: {site_name} #####")
        results[row["href"]] = inspect_parent_or_standalone_like_page(
            driver=driver,
            url=row["href"],
            site_name=site_name
        )

    return results


# 페이지 1개 구조 확인 공통: 상위든 하위든 일단 한 페이지 자체 구조 확인
def inspect_parent_or_standalone_like_page(driver, url: str, site_name: str = None, max_links: int = 80):
    title, full_text, soup = load_page(driver, url)
    clean_main = extract_clean_main_text(driver)
    selector_df = inspect_main_candidates(driver)

    links = []
    for a in soup.find_all("a", href=True):
        href = normalize_url(urljoin(url, a["href"]))
        link_text = a.get_text(" ", strip=True)

        if not link_text:
            continue
        if not is_same_domain(href, url):
            continue

        links.append({
            "site_name": site_name,
            "link_text": link_text,
            "href": href
        })

    df_links = pd.DataFrame(links).drop_duplicates()

    print("=" * 120)
    print("[페이지 구조 확인]")
    print("SITE NAME :", site_name)
    print("URL       :", url)
    print("TITLE     :", title)
    print("DEAD PAGE :", is_dead_page(title, full_text))

    print("\n[본문 selector 후보]")
    print(selector_df if not selector_df.empty else "후보 없음")

    print("\n[본문만 추출 결과]")
    print("selector      :", clean_main["selector"])
    print("raw_text_len  :", clean_main["raw_text_len"])
    print("clean_text_len:", clean_main["clean_text_len"])
    print("sample        :", clean_main["text"][:400])

    print("\n[내부 링크 샘플]")
    print(df_links.head(max_links) if not df_links.empty else "내부 링크 없음")
    print("=" * 120)

    return {
        "title": title,
        "full_text": full_text,
        "clean_main": clean_main,
        "selector_df": selector_df,
        "links_df": df_links
    }

In [85]:
# 공유 사이트형 목록
shared_sites = [
    {

        "site_name": "연세대 생활과학대학",
        "url": "https://che.yonsei.ac.kr/che/index.do",
        "dept_keywords": ["의류환경", "식품영양", "실내건축", "아동가족", "통합디자인"]
    },
    {
        "site_name": "경희대 생활과학대학",
        "url": "https://che.khu.ac.kr/che/user/main/view.do",
        "dept_keywords": ["아동가족", "주거환경", "의상", "식품영양"]
    },
    {
        "site_name": "이화여대 조형예술대학",
        "url": "https://artndesign.ewha.ac.kr/artewha/index.do",
        "dept_keywords": ["섬유패션", "섬유예술"]
    },

    {
        "site_name": "서울여대 대학 페이지(식품영양/패션/마이크로전공)",
        "url": "https://www.swu.ac.kr/www/natusciencej_1.html",
        "dept_keywords": ["식품영양", "패션산업", "아동청소년복지", "스마트영양관리", "식품개발실무"]
    },

    {
        "site_name": "경인교육대학교 생활과학교육과",
        "url": "https://edu.ginue.ac.kr/practical/Main.do",
        "dept_keywords": ["생활과학교육", "과학교육과"]
    }
]

In [86]:
standalone_sites = [
    # 서울대
    ("서울대 생활과학대학", "https://che.snu.ac.kr/"), # 생과대 전용

    ("서울대 식품영양학과", "https://foodnutrition.snu.ac.kr/"),
    ("서울대 의류학과", "https://clothing.snu.ac.kr/"),
    ("서울대 아동가족학과", "https://childfamily.snu.ac.kr/"),
    ("서울대 소비자학과", "https://consumer.snu.ac.kr/"),


    # 고려대
    ("고려대 식품자원경제학과", "https://frecon.korea.ac.kr/frecon/index.do"), 
    ("고려대 식품공학과", "https://foodscience.korea.ac.kr/foodscience/index.do"),
    ("고려대 패션디자인 및 머천다이징 전공", "https://fadm.korea.ac.kr/kufadm/index.do"),
    ("고려대 가정교육과", "https://homedu.korea.ac.kr/homedu/index.do#none"),
    ("고려대 지속가능생활시스템융합전공", "https://slp.korea.ac.kr/slp/index.do"),
    ("고려대 교육대학원 가정교육 전공", "https://edugrad.korea.ac.kr/edugrad/master/master_major.do"),

    # 이화여대
    ("이화여대 식품생명공학과", "https://foodbiotech.ewha.ac.kr/foodbiotech/index.do"),
    ("이화여대 건축학과", "https://ea.ewha.ac.kr/"),
    ("이화여대 식품영양학과", "https://nsfm.ewha.ac.kr/nsfm/index.do"),
    ("이화여대 의류산업학과", "https://fashion.ewha.ac.kr/fashion/index.do"),
    ("이화여대 소비자학과", "https://consume.ewha.ac.kr/consume/index.do"),
    ("이화여대 아동학과", "https://child.ewha.ac.kr/child/index.do"),
    ("이화여대 교육대학원 가정과교육", "https://ged.ewha.ac.kr/gedewha/index.do"),
    ("이화여대 디자인대학원", "https://gsd.ewha.ac.kr/ewhagsd/index.do"),

    # 성균관대
    ("성균관대 식품생명공학과", "https://skb.skku.edu/foodlife/index.do"),
    ("성균관대 의상학과", "https://fashion.skku.edu/fashion/index.do"),
    ("성균관대 아동청소년학과", "https://child.skku.edu/child/index.do"),
    ("성균관대 생활과학연구소", "https://swb.skku.edu/hls/index.do"),
    ("성균관대 소비자학과", "https://cf.skku.edu/skkucf/index.do"),

    # 한양대
    ("한양대 식품영양학과", "https://fn.hanyang.ac.kr/home"),
    ("한양대 의류학과", "https://fashion.hanyang.ac.kr/home"),

    # 중앙대
    ("중앙대 식품공학부 식품영양전공", "https://cobiotech.cau.ac.kr/foodnutri/"),
    ("중앙대 식품공학전공", "https://cobiotech.cau.ac.kr/foodtech/"),
    ("중앙대 패션전공", "https://fashion.cau.ac.kr/"),
    ("중앙대 실내환경디자인전공", "https://www.indesign.cau.ac.kr/"),

    # 건국대
    ("건국대 의상디자인학과", "https://www.konkuk.ac.kr/apparel/index.do"),
    ("건국대 리빙디자인학과", "https://livingdesign.konkuk.ac.kr/livingdesign/index.do"),
    ("건국대 K뷰티산업융합학과", "https://kbeauty.konkuk.ac.kr/kbeauty/index.do"),
    ("건국대 식품유통공학과", "https://kufsm.konkuk.ac.kr/kufsm/index.do"),
    ("건국대 식량자원과학과", "https://cropscience.konkuk.ac.kr/cropscience/index.do"),
    ("건국대 동물자원식품과학 전공", "https://anis.konkuk.ac.kr/anis/index.do"),

    # 숙명여대
    ("숙명여대 가족자원경영학과", "https://family.sookmyung.ac.kr/family/index.do"),
    ("숙명여대 식품영양학과", "https://fn.sookmyung.ac.kr/fn/index.do"),
    ("숙명여대 의류학과", "https://cloth.sookmyung.ac.kr/cloth/index.do"),
    ("숙명여대 아동복지학부", "https://childwelfare.sookmyung.ac.kr/childwelfare/index.do"),
    ("숙명여대 소비자경제학과", "https://conecon.sookmyung.ac.kr/"),

    # 동덕여대
    ("동덕여대 패션디자인전공", "https://fashion.dongduk.ac.kr/"),
    ("동덕여대 아동학과", "https://child.dongduk.ac.kr/"),
    ("동덕여대 식품영양학과", "https://nutrition.dongduk.ac.kr/"),
    ("동덕여대 화장품학전공", "https://cosmetics.dongduk.ac.kr/"),

    # 서울여대
    ("서울여대 아동학과", "https://childstudy.swu.ac.kr/"),

    # 덕성여대
    ("덕성여대 아동가족학전공", "https://www.duksung.ac.kr/hdfs/main.do"),
    ("덕성여대 의상디자인전공", "https://www.duksung.ac.kr/fashion/main.do"),
    ("덕성여대 식품영양학전공", "https://www.duksung.ac.kr/fan/main.do"),
    ("덕성여대 실내디자인전공", "https://www.duksung.ac.kr/interior/main.do"),
    ("덕성여대 텍스타일디자인전공", "https://www.duksung.ac.kr/textile/main.do"),

    # 인하대
    ("인하대 식품영양학과", "https://foodnutri.inha.ac.kr/foodnutri/index.do"),
    ("인하대 소비자학과", "https://consumer.inha.ac.kr/consumer/index.do"),
    ("인하대 아동심리학과", "https://child.inha.ac.kr/child/index.do"),
    ("인하대 의류디자인학과", "https://fashion.inha.ac.kr/fashion/index.do"),
    ("인하대 바이오식품공학과", "https://foodscience.inha.ac.kr/foodscience/index.do"),

    # 단국대
    ("단국대 식품영양학과", "https://cms.dankook.ac.kr/web/foodnutrition"),
    ("단국대 식품공학과", "https://cms.dankook.ac.kr/web/food"),
    ("단국대 식품자원경제학과", "https://cms.dankook.ac.kr/web/ere"),
    ("단국대 패션산업디자인전공", "https://cms.dankook.ac.kr/web/fashion"),

    # 교원대 / 교대
    ("한국교원대 가정교육과", "https://www.knue.ac.kr/homeedu/index.do"),
    ("서울교대 초등생활과학교육(석사)", "https://grad.snue.ac.kr/grad/cm/cntnts/cntntsView.do?mi=3167&cntntsId=3122"),
    ("서울교대 초등생활과학/컴퓨터교육(박사)", "https://grad.snue.ac.kr/grad/cm/cntnts/cntntsView.do?mi=3241&cntntsId=3181"),
    ("경인교대 생활과학교육과", "https://edu.ginue.ac.kr/practical/Main.do"),
]

In [87]:
if __name__ == "__main__":
    driver = create_driver(headless=False)

    try:
        print("\n" + "#" * 40 + " 공유 사이트형 구조 확인 " + "#" * 40)

        for site in shared_sites:
            print(f"\n\n===== 상위 홈페이지 확인: {site['site_name']} =====")

            # 1) 상위 홈페이지 자체 구조 확인
            inspect_parent_site(
                driver=driver,
                site_name=site["site_name"],
                url=site["url"],
                dept_keywords=site["dept_keywords"],
                max_links=100
            )

            # 2) 상위 홈페이지 내부 학과/전공 하위 페이지 구조 확인
            inspect_child_pages_from_parent(
                driver=driver,
                parent_url=site["url"],
                dept_keywords=site["dept_keywords"],
                max_child_pages=10
            )

        print("\n" + "#" * 40 + " 독립 사이트형 구조 확인 " + "#" * 40)

        for site_name, url in standalone_sites:
            print(f"\n\n===== 독립 사이트 확인: {site_name} =====")

            inspect_parent_or_standalone_like_page(
                driver=driver,
                url=url,
                site_name=site_name,
                max_links=80
            )

    finally:
        driver.quit()


######################################## 공유 사이트형 구조 확인 ########################################


===== 상위 홈페이지 확인: 연세대 생활과학대학 =====
[상위 홈페이지 구조 확인]
SITE NAME : 연세대 생활과학대학
URL       : https://che.yonsei.ac.kr/che/index.do
TITLE     : 연세대학교 생활과학대학
DEAD PAGE : False

[본문 selector 후보]
  selector  text_len                                             sample
0     body      1139  홈으로 수강신청 사이트맵 연세대학교 홈페이지 English 대학 소개 학과소개 입학...

[본문만 추출 결과]
selector      : None
raw_text_len  : 0
clean_text_len: 0
sample        : 

[학과/전공 키워드 매칭 링크]
      site_name dept_match                                          link_text  \
14   연세대 생활과학대학       의류환경                                             의류환경학과   
15   연세대 생활과학대학       식품영양                                             식품영양학과   
16   연세대 생활과학대학       실내건축                                             실내건축학과   
18   연세대 생활과학대학      통합디자인                                            통합디자인학과   
43   연세대 생활과학대학       의류환경                                   

In [138]:
khu_sites = [
    {
        "site_name": "경희대 생활과학대학",
        "url": "https://che.khu.ac.kr/che/user/main/view.do",
        "dept_keywords": ["아동가족", "주거환경", "의상", "식품영양"]
    },
]

In [141]:
if __name__ == "__main__":
    driver = create_driver(headless=False)

    try:
        print("\n" + "#" * 40 + " khu 구조 확인 " + "#" * 40)

        for site in khu_sites:
            print(f"\n\n===== 상위 홈페이지 확인: {site['site_name']} =====")

            # 1) 상위 홈페이지 자체 구조 확인
            inspect_parent_site(
                driver=driver,
                site_name=site["site_name"],
                url=site["url"],
                dept_keywords=site["dept_keywords"],
                max_links=100
            )

            # 2) 상위 홈페이지 내부 학과/전공 하위 페이지 구조 확인
            inspect_child_pages_from_parent(
                driver=driver,
                parent_url=site["url"],
                dept_keywords=site["dept_keywords"],
                max_child_pages=10
            )

    finally:
        driver.quit()


######################################## khu 구조 확인 ########################################


===== 상위 홈페이지 확인: 경희대 생활과학대학 =====
[상위 홈페이지 구조 확인]
SITE NAME : 경희대 생활과학대학
URL       : https://che.khu.ac.kr/che/user/main/view.do
TITLE     : 생활과학대학
DEAD PAGE : False

[본문 selector 후보]
  selector  text_len                                             sample
0    .cont       262  1. 평화의전당 대관과 관련하여 주차통제를 안내하오니 구성원 여러분의 협조 부탁드립...
1     .txt       114  KYUNGHEE UNIVERSITY COLLEGE OF HUMAN ECOLOGY 경...
2     body      2150  로그인 KHU Home Webmail 검색창 열기 생활과학대학 아동가족학과 주거환경...

[본문만 추출 결과]
selector      : .cont
raw_text_len  : 262
clean_text_len: 262
sample        : 1. 평화의전당 대관과 관련하여 주차통제를 안내하오니 구성원 여러분의 협조 부탁드립니다.  가. 행사명 : K트롯 어워즈(가제)        1) 대관일정 : 2026.04.09.(목) ~ 2026.04.12.(일)        2) 주    최 : 주식회사 팬틱스  나. 주차통제       1) 부분통제 : 2026.04.10.(금) 06:00 ~ 24:00       2) 전면통제 : 2026.04.11.(토) 06:00 ~ 2026.04.12.(일) 01:00

[학과/전공 키워드 매칭 링크]
     site_name dept_match link_text  \
16  경희대 생활과학대학      

# 크롤링

In [126]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qs
from collections import deque

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


# 기본 설정
CURRENT_YEAR = datetime.now().year
NOTICE_YEAR_LIMIT = CURRENT_YEAR - 3
MIN_TEXT_LEN = 50

In [127]:
# 드라이버
def create_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1200")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


# 유틸
def normalize_url(url: str) -> str:
    parsed = urlparse(url)
    return parsed._replace(fragment="").geturl()

def is_same_domain(url: str, base_url: str) -> bool:
    return urlparse(url).netloc == urlparse(base_url).netloc

def get_menu_no(url: str):
    try:
        qs = parse_qs(urlparse(url).query)
        return qs.get("menuNo", [None])[0]
    except Exception:
        return None

def should_visit_url(url: str) -> bool:
    url_lower = (url or "").lower()

    blocked_ext = [
        ".jpg", ".jpeg", ".png", ".gif", ".svg", ".pdf", ".doc", ".docx",
        ".xls", ".xlsx", ".zip", ".hwp", ".mp4", ".avi", ".mov", ".wmv"
    ]
    if any(url_lower.endswith(ext) for ext in blocked_ext):
        return False

    blocked_scheme = ["mailto:", "javascript:", "tel:"]
    if any(x in url_lower for x in blocked_scheme):
        return False

    return True


# dead page / 영문 페이지 / 노이즈
def is_dead_page(title: str, text: str) -> bool:
    combined = f"{title}\n{text}".lower()
    patterns = [
        "요청하신 페이지가 존재하지 않습니다",
        "페이지를 찾을 수 없습니다",
        "존재하지 않습니다",
        "404",
        "not found",
    ]
    return any(p.lower() in combined for p in patterns)

def is_english_page(url: str, title: str, text: str) -> bool:
    url_lower = (url or "").lower()
    title_lower = (title or "").lower()

    url_patterns = ["/eng/", "/english/", "/en/", "lang=en", "locale=en"]
    if any(p in url_lower for p in url_patterns):
        return True

    title_patterns = [
        "department of", "college of", "faculty", "graduate school",
        "curriculum", "admission", "research"
    ]
    if any(p in title_lower for p in title_patterns):
        return True

    sample = (text or "")[:1200]
    korean_chars = len(re.findall(r"[가-힣]", sample))
    english_words = len(re.findall(r"[A-Za-z]{2,}", sample))

    if korean_chars < 20 and english_words >= 35:
        return True

    return False

NOISE_PATTERNS = [
    "로그인", "Login", "사이트맵", "SITEMAP", "English", "KUPID", "HY-in",
    "전체메뉴", "본문 바로가기", "주메뉴 바로가기", "서브메뉴 바로가기",
    "푸터 바로가기", "내용으로 건너뛰기", "교내주요사이트", "개인정보처리방침",
    "법적고지", "TOP", "검색", "팝업 닫기", "1일간팝업열지않기", "1주일간 보지않기"
]

def clean_text(text: str) -> str:
    cleaned = text
    for noise in NOISE_PATTERNS:
        cleaned = cleaned.replace(noise, "")
    cleaned = re.sub(r"\n{2,}", "\n", cleaned)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    return cleaned.strip()



In [128]:
# 날짜 / page_type
def extract_year_from_text(text: str):
    match = re.search(r"(20\d{2}|19\d{2})", text or "")
    if match:
        year = int(match.group(1))
        if 1900 <= year <= CURRENT_YEAR + 1:
            return year
    return None

def is_recent_notice(title: str, text: str, url: str) -> bool:
    combined = f"{title}\n{text}\n{url}"
    year = extract_year_from_text(combined)
    if year is None:
        return True
    return year >= NOTICE_YEAR_LIMIT

def classify_page_type(title: str, url: str, text: str) -> str:
    combined = f"{title}\n{url}\n{text}".lower()

    if any(k in combined for k in ["공지", "notice", "게시판", "board", "news", "자료실", "faq", "q&a"]):
        return "notice"

    rules = [
        ("faculty_index", ["교수진", "교수소개", "전임교수", "명예교수", "faculty"]),
        ("research", ["연구", "연구실", "논문", "project", "research", "lab"]),
        ("curriculum", ["교육과정", "교과과정", "커리큘럼", "curriculum"]),
        ("course", ["교과목", "과목소개", "강의", "course"]),
        ("requirements", ["졸업요건", "이수기준", "학사규정", "내규", "시행세칙"]),
        ("program", ["복수전공", "연계", "융합", "협동", "program"]),
        ("career", ["진로", "취업", "자격증", "졸업 후 진로", "졸업후진로"]),
        ("student", ["학생활동", "학생", "졸업생", "인터뷰"]),
        ("activity", ["행사", "세미나", "심포지엄", "전시", "갤러리"]),
        ("about", ["학과소개", "전공소개", "소개", "연혁", "교육목표", "인사말", "오시는길"]),
    ]

    for page_type, keywords in rules:
        if any(k.lower() in combined for k in keywords):
            return page_type

    return "etc"


# 본문 추출: 구조 로그 반영: .txt 푸터를 피하고, article/main/content를 우선
def score_text_block(text: str, selector: str) -> int:
    if not text:
        return -9999

    score = 0
    length = len(text)
    score += min(length, 4000)

    # 좋은 selector 가점
    if selector in ["main", "#content", ".content", ".contents", ".sub_content", "article", ".cont", ".board_view", ".view", ".con_box"]:
        score += 500

    # 너무 짧으면 감점
    if length < 120:
        score -= 800

    # footer성 텍스트 감점
    footer_keywords = [
        "COPYRIGHT", "ALL RIGHT RESERVED", "서울캠퍼스", "국제캠퍼스",
        "광릉캠퍼스", "개인정보처리방침", "법적고지"
    ]
    if any(k.lower() in text.lower() for k in footer_keywords):
        score -= 1500

    # 메뉴성 텍스트 과다 감점
    menu_keywords = ["로그인", "사이트맵", "English", "HOME", "KUPID", "HY-in"]
    menu_hits = sum(1 for k in menu_keywords if k.lower() in text.lower())
    score -= 120 * menu_hits

    # 문장성 가점
    korean_chars = len(re.findall(r"[가-힣]", text))
    if korean_chars >= 80:
        score += 400

    return score

def extract_main_text(driver):
    candidate_selectors = [
        "main", "#content", ".content", ".contents", ".sub_content",
        "article", ".cont", ".board_view", ".view", ".con_box",
        ".entry-content", ".txt", "body"
    ]

    candidates = []

    for sel in candidate_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            elems = []

        for elem in elems[:3]:
            try:
                raw = elem.text.strip()
                if not raw:
                    continue
                cleaned = clean_text(raw)
                score = score_text_block(cleaned, sel)
                candidates.append({
                    "selector": sel,
                    "raw": raw,
                    "cleaned": cleaned,
                    "score": score
                })
            except Exception:
                continue

    if not candidates:
        return None, ""

    best = sorted(candidates, key=lambda x: x["score"], reverse=True)[0]
    if len(best["cleaned"]) < MIN_TEXT_LEN:
        return best["selector"], ""

    return best["selector"], best["cleaned"]


# 링크 수집
def collect_internal_links(driver, current_url: str, base_url: str,
                           allowed_prefixes=None, allowed_menu_nos=None):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    found = set()

    for a in soup.find_all("a", href=True):
        href = normalize_url(urljoin(current_url, a["href"]))
        if not should_visit_url(href):
            continue
        if not is_same_domain(href, base_url):
            continue

        if allowed_prefixes:
            if not any(href.startswith(p) for p in allowed_prefixes):
                continue

        if allowed_menu_nos:
            menu_no = get_menu_no(href)
            if menu_no and menu_no not in allowed_menu_nos:
                continue

        found.add(href)

    return list(found)


In [129]:
# 공통 크롤링 함수
def crawl_pages(driver, university, college, department, seed_url,
                allowed_prefixes=None, allowed_menu_nos=None,
                max_pages=150, max_depth=2):
    base_domain = urlparse(seed_url).netloc
    visited = set()
    queue = deque([(normalize_url(seed_url), 0)])
    rows = []

    while queue and len(visited) < max_pages:
        current_url, depth = queue.popleft()

        if current_url in visited:
            continue
        if depth > max_depth:
            continue

        try:
            driver.get(current_url)
            time.sleep(1.3)
            visited.add(current_url)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            title = soup.title.get_text(" ", strip=True) if soup.title else ""

            selector_used, text = extract_main_text(driver)
            full_text = soup.get_text("\n", strip=True)

            if is_dead_page(title, full_text):
                continue

            if is_english_page(current_url, title, text or full_text):
                continue

            if text and len(text) >= MIN_TEXT_LEN:
                page_type = classify_page_type(title, current_url, text)

                if page_type == "notice":
                    if is_recent_notice(title, text, current_url):
                        rows.append({
                            "university": university,
                            "college": college,
                            "department": department,
                            "page_type": page_type,
                            "url": current_url,
                            "title": title,
                            "text": text,
                            "selector_used": selector_used,
                        })
                else:
                    rows.append({
                        "university": university,
                        "college": college,
                        "department": department,
                        "page_type": page_type,
                        "url": current_url,
                        "title": title,
                        "text": text,
                        "selector_used": selector_used,
                    })

            next_links = collect_internal_links(
                driver=driver,
                current_url=current_url,
                base_url=f"https://{base_domain}",
                allowed_prefixes=allowed_prefixes,
                allowed_menu_nos=allowed_menu_nos,
            )

            for link in next_links:
                if link not in visited:
                    queue.append((link, depth + 1))

        except Exception as e:
            print(f"[ERROR] {current_url} | {e}")

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df


# 독립 사이트형 wrapper
def crawl_standalone_site(driver, university, college, department, seed_url,
                          max_pages=150, max_depth=2):
    return crawl_pages(
        driver=driver,
        university=university,
        college=college,
        department=department,
        seed_url=seed_url,
        allowed_prefixes=[normalize_url(seed_url).split("/", 3)[0] + "//" + urlparse(seed_url).netloc],
        allowed_menu_nos=None,
        max_pages=max_pages,
        max_depth=max_depth
    )


# 공유 사이트형 wrapper: prefix 방식 또는 menuNo 방식 선택
def crawl_shared_site_by_prefix(driver, university, college, department, seed_url,
                                allowed_prefixes, max_pages=150, max_depth=2):
    return crawl_pages(
        driver=driver,
        university=university,
        college=college,
        department=department,
        seed_url=seed_url,
        allowed_prefixes=allowed_prefixes,
        allowed_menu_nos=None,
        max_pages=max_pages,
        max_depth=max_depth
    )

def crawl_shared_site_by_menu(driver, university, college, department, seed_url,
                              allowed_menu_nos, max_pages=150, max_depth=3):
    return crawl_pages(
        driver=driver,
        university=university,
        college=college,
        department=department,
        seed_url=seed_url,
        allowed_prefixes=None,
        allowed_menu_nos=set(allowed_menu_nos),
        max_pages=max_pages,
        max_depth=max_depth
    )


In [92]:
# 예시 실행
if __name__ == "__main__":
    driver = create_driver(headless=False)

    try:
        # 1) 독립 사이트형 예시: 한양대 식품영양학과
        df_fn = crawl_standalone_site(
            driver=driver,
            university="한양대학교",
            college="생활과학대학",
            department="식품영양학과",
            seed_url="https://fn.hanyang.ac.kr/home",
            max_pages=80,
            max_depth=2
        )
        print("\n[한양대 식품영양학과]")
        print(df_fn[["page_type", "title", "selector_used", "url"]].head(20))
        df_fn.to_csv("hanyang_fn_text_pages.csv", index=False, encoding="utf-8-sig")

        # 2) 공유 사이트형 예시: 연세대 식품영양학과
        df_yonsei_food = crawl_shared_site_by_prefix(
            driver=driver,
            university="연세대학교",
            college="생활과학대학",
            department="식품영양학과",
            seed_url="https://che.yonsei.ac.kr/che/departments/food_intro.do",
            allowed_prefixes=[
                "https://che.yonsei.ac.kr/che/departments/food_",
                "https://che.yonsei.ac.kr/che/food/"
            ],
            max_pages=100,
            max_depth=2
        )
        print("\n[연세대 식품영양학과]")
        print(df_yonsei_food[["page_type", "title", "selector_used", "url"]].head(20))
        df_yonsei_food.to_csv("yonsei_food_text_pages.csv", index=False, encoding="utf-8-sig")

        # 3) 공유 사이트형 예시: 경희대 식품영양학과(menuNo 기반)
        df_khu_food = crawl_shared_site_by_menu(
            driver=driver,
            university="경희대학교",
            college="생활과학대학",
            department="식품영양학과",
            seed_url="https://che.khu.ac.kr/che/user/contents/view.do?menuNo=12700014",
            allowed_menu_nos=[
                "12700014",   # 학과소개
                "12700065",   # 교수소개 계열 예시: 실제 값은 구조 확인 후 보정
                "12700066",
                "12700067",
                "12700068"
            ],
            max_pages=120,
            max_depth=3
        )
        print("\n[경희대 식품영양학과]")
        print(df_khu_food[["page_type", "title", "selector_used", "url"]].head(20))
        df_khu_food.to_csv("khu_food_text_pages.csv", index=False, encoding="utf-8-sig")

    finally:
        driver.quit()


[한양대 식품영양학과]
        page_type                             title   selector_used  \
0          notice                홈 - 식품영양학과 - 한양대학교        #content   
1          notice         대학원 공지사항 - 식품영양학과 - 한양대학교        #content   
2          notice         대학원 공지사항 - 식품영양학과 - 한양대학교  .entry-content   
3        research      식품분자미생물 연구실 - 식품영양학과 - 한양대학교        #content   
4          notice          학부 공지사항 - 식품영양학과 - 한양대학교  .entry-content   
5        research  기능성식품 및 식품가공연구실 - 식품영양학과 - 한양대학교        #content   
6      curriculum              학생회 - 식품영양학과 - 한양대학교        #content   
7        research       임상영양과대사실험실 - 식품영양학과 - 한양대학교        #content   
8          notice          학부 공지사항 - 식품영양학과 - 한양대학교        #content   
9   faculty_index          졸업 후 진로 - 식품영양학과 - 한양대학교        #content   
10         notice             채용공고 - 식품영양학과 - 한양대학교        #content   
11     curriculum             졸업요건 - 식품영양학과 - 한양대학교        #content   
12       research    건강기능영양유전체학연구실 - 식품영양학과 - 한양대학교        #con

## main

In [ ]:
# 공유 사이트형 설정
shared_config = [
    # 연세대
    {
        "university": "연세대학교",
        "college": "생활과학대학",
        "department": "의류환경학과",
        "seed_url": "https://che.yonsei.ac.kr/che/departments/clothing_intro.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://che.yonsei.ac.kr/che/departments/clothing_",
            "https://che.yonsei.ac.kr/che/departments/clothing",
        ],
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "연세대학교",
        "college": "생활과학대학",
        "department": "식품영양학과",
        "seed_url": "https://che.yonsei.ac.kr/che/departments/food_intro.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://che.yonsei.ac.kr/che/departments/food_",
            "https://che.yonsei.ac.kr/che/food/"
        ],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "연세대학교",
        "college": "생활과학대학",
        "department": "실내건축학과",
        "seed_url": "https://che.yonsei.ac.kr/che/interior/interior_intro01.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://che.yonsei.ac.kr/che/interior/"
        ],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "연세대학교",
        "college": "생활과학대학",
        "department": "아동가족학과",
        "seed_url": "https://che.yonsei.ac.kr/che/child/child_intro01.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://che.yonsei.ac.kr/che/child/"
        ],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "연세대학교",
        "college": "생활과학대학",
        "department": "통합디자인학과",
        "seed_url": "https://che.yonsei.ac.kr/che/design_intro01.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://che.yonsei.ac.kr/che/design",
            "https://che.yonsei.ac.kr/che/design_"
        ],
        "max_pages": 100,
        "max_depth": 2,
    },

    # 경희대
    {
        "university": "경희대학교",
        "college": "생활과학대학",
        "department": "아동가족학과",
        "seed_url": "https://che.khu.ac.kr/che/user/contents/view.do?menuNo=12700011",
        "mode": "menu",
        "allowed_menu_nos": ["12700011"],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "경희대학교",
        "college": "생활과학대학",
        "department": "주거환경학과",
        "seed_url": "https://che.khu.ac.kr/che/user/contents/view.do?menuNo=12700012",
        "mode": "menu",
        "allowed_menu_nos": ["12700012"],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "경희대학교",
        "college": "생활과학대학",
        "department": "의상학과",
        "seed_url": "https://che.khu.ac.kr/che/user/contents/view.do?menuNo=12700013",
        "mode": "menu",
        "allowed_menu_nos": ["12700013", "12700064"],
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "경희대학교",
        "college": "생활과학대학",
        "department": "식품영양학과",
        "seed_url": "https://che.khu.ac.kr/che/user/contents/view.do?menuNo=12700014",
        "mode": "menu",
        "allowed_menu_nos": ["12700014"],
        "max_pages": 120,
        "max_depth": 2,
    },

    # 이화여대 조형예술대학
    {
        "university": "이화여자대학교",
        "college": "조형예술대학",
        "department": "섬유패션학부",
        "seed_url": "https://artndesign.ewha.ac.kr/artewha/55/subview.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://artndesign.ewha.ac.kr/artewha/55/",
            "https://artndesign.ewha.ac.kr/artewha/116/",
            "https://artndesign.ewha.ac.kr/artewha/117/"
        ],
        "max_pages": 120,
        "max_depth": 2,
    },

    # 서울여대 대학 페이지형
    {
        "university": "서울여자대학교",
        "college": "자연과학대학",
        "department": "식품영양학과",
        "seed_url": "https://www.swu.ac.kr/www/natusciencej_1.html",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://www.swu.ac.kr/www/natusciencej_1.html"
        ],
        "max_pages": 40,
        "max_depth": 1,
    },
    {
        "university": "서울여자대학교",
        "college": "미래산업융합대학",
        "department": "패션산업학과",
        "seed_url": "https://www.swu.ac.kr/www/futureb_1.html",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://www.swu.ac.kr/www/futureb_1.html"
        ],
        "max_pages": 40,
        "max_depth": 1,
    },
    {
        "university": "서울여자대학교",
        "college": "마이크로전공",
        "department": "스마트영양관리",
        "seed_url": "https://www.swu.ac.kr/www/microj_1.html",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://www.swu.ac.kr/www/microj_1.html"
        ],
        "max_pages": 40,
        "max_depth": 1,
    },
    {
        "university": "서울여자대학교",
        "college": "마이크로전공",
        "department": "(융합)식품개발실무",
        "seed_url": "https://www.swu.ac.kr/www/microw_1.html",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://www.swu.ac.kr/www/microw_1.html"
        ],
        "max_pages": 40,
        "max_depth": 1,
    },

    # 경인교육대
    {
        "university": "경인교육대학교",
        "college": "생활과학교육과",
        "department": "생활과학교육과",
        "seed_url": "https://edu.ginue.ac.kr/practical/Main.do",
        "mode": "prefix",
        "allowed_prefixes": [
            "https://edu.ginue.ac.kr/practical/",
            "http://edu.ginue.ac.kr/practical/"
        ],
        "max_pages": 100,
        "max_depth": 2,
    },
]

In [97]:
# 독립 사이트형 설정
standalone_config = [
    # 서울대
    {
        "university": "서울대학교",
        "college": "생활과학대학",
        "department": "생활과학대학",
        "seed_url": "https://che.snu.ac.kr/",
        "max_pages": 80,
        "max_depth": 2,
    },
    {
        "university": "서울대학교",
        "college": "생활과학대학",
        "department": "식품영양학과",
        "seed_url": "https://foodnutrition.snu.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "서울대학교",
        "college": "생활과학대학",
        "department": "의류학과",
        "seed_url": "https://clothing.snu.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "서울대학교",
        "college": "생활과학대학",
        "department": "아동가족학과",
        "seed_url": "https://childfamily.snu.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "서울대학교",
        "college": "생활과학대학",
        "department": "소비자학과",
        "seed_url": "https://consumer.snu.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 고려대
    {
        "university": "고려대학교",
        "college": "생명과학대학",
        "department": "식품자원경제학과",
        "seed_url": "https://frecon.korea.ac.kr/frecon/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "고려대학교",
        "college": "생명과학대학",
        "department": "식품공학과",
        "seed_url": "https://foodscience.korea.ac.kr/foodscience/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "고려대학교",
        "college": "디자인조형학부",
        "department": "패션디자인 및 머천다이징 전공",
        "seed_url": "https://fadm.korea.ac.kr/kufadm/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "고려대학교",
        "college": "사범대학",
        "department": "가정교육과",
        "seed_url": "https://homedu.korea.ac.kr/homedu/index.do#none",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "고려대학교",
        "college": "대학원",
        "department": "지속가능생활시스템융합전공",
        "seed_url": "https://slp.korea.ac.kr/slp/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "고려대학교",
        "college": "교육대학원",
        "department": "가정교육 전공",
        "seed_url": "https://edugrad.korea.ac.kr/edugrad/master/master_major.do",
        "max_pages": 60,
        "max_depth": 1,
    },

    # 이화여대
    {
        "university": "이화여자대학교",
        "college": "공과대학",
        "department": "식품생명공학과",
        "seed_url": "https://foodbiotech.ewha.ac.kr/foodbiotech/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "공과대학",
        "department": "건축학과",
        "seed_url": "https://ea.ewha.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "신산업융합대학",
        "department": "식품영양학과",
        "seed_url": "https://nsfm.ewha.ac.kr/nsfm/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "신산업융합대학",
        "department": "의류산업학과",
        "seed_url": "https://fashion.ewha.ac.kr/fashion/index.do",
        "max_pages": 140,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "사회과학대학",
        "department": "소비자학과",
        "seed_url": "https://consume.ewha.ac.kr/consume/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "일반대학원",
        "department": "아동학과",
        "seed_url": "https://child.ewha.ac.kr/child/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "교육대학원",
        "department": "가정과교육",
        "seed_url": "https://ged.ewha.ac.kr/gedewha/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "이화여자대학교",
        "college": "디자인대학원",
        "department": "디자인대학원",
        "seed_url": "https://gsd.ewha.ac.kr/ewhagsd/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },

    # 성균관대
    {
        "university": "성균관대학교",
        "college": "생명공학대학",
        "department": "식품생명공학과",
        "seed_url": "https://skb.skku.edu/foodlife/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "성균관대학교",
        "college": "예술대학",
        "department": "의상학과",
        "seed_url": "https://fashion.skku.edu/fashion/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "성균관대학교",
        "college": "사회과학대학",
        "department": "아동청소년학과",
        "seed_url": "https://child.skku.edu/child/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "성균관대학교",
        "college": "생활과학연구소",
        "department": "생활과학연구소",
        "seed_url": "https://swb.skku.edu/hls/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "성균관대학교",
        "college": "사회과학대학",
        "department": "소비자학과",
        "seed_url": "https://cf.skku.edu/skkucf/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 한양대
    {
        "university": "한양대학교",
        "college": "생활과학대학",
        "department": "식품영양학과",
        "seed_url": "https://fn.hanyang.ac.kr/home",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "한양대학교",
        "college": "생활과학대학",
        "department": "의류학과",
        "seed_url": "https://fashion.hanyang.ac.kr/home",
        "max_pages": 100,
        "max_depth": 2,
    },

    # 중앙대
    {
        "university": "중앙대학교",
        "college": "생명공학대학",
        "department": "식품공학부 식품영양전공",
        "seed_url": "https://cobiotech.cau.ac.kr/foodnutri/",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "중앙대학교",
        "college": "생명공학대학",
        "department": "식품공학전공",
        "seed_url": "https://cobiotech.cau.ac.kr/foodtech/",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "중앙대학교",
        "college": "예술대학",
        "department": "패션전공",
        "seed_url": "https://fashion.cau.ac.kr/",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "중앙대학교",
        "college": "예술대학",
        "department": "실내환경디자인전공",
        "seed_url": "https://www.indesign.cau.ac.kr/",
        "max_pages": 100,
        "max_depth": 2,
    },

    # 건국대
    {
        "university": "건국대학교",
        "college": "예술디자인대학",
        "department": "의상디자인학과",
        "seed_url": "https://www.konkuk.ac.kr/apparel/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "건국대학교",
        "college": "예술디자인대학",
        "department": "리빙디자인학과",
        "seed_url": "https://livingdesign.konkuk.ac.kr/livingdesign/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "건국대학교",
        "college": "공과대학",
        "department": "K뷰티산업융합학과",
        "seed_url": "https://kbeauty.konkuk.ac.kr/kbeauty/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "건국대학교",
        "college": "상허생명과학대학",
        "department": "식품유통공학과",
        "seed_url": "https://kufsm.konkuk.ac.kr/kufsm/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "건국대학교",
        "college": "상허생명과학대학",
        "department": "식량자원과학과",
        "seed_url": "https://cropscience.konkuk.ac.kr/cropscience/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },
    {
        "university": "건국대학교",
        "college": "생명과학대학",
        "department": "동물자원식품과학 전공",
        "seed_url": "https://anis.konkuk.ac.kr/anis/index.do",
        "max_pages": 100,
        "max_depth": 2,
    },

    # 숙명여대
    {
        "university": "숙명여자대학교",
        "college": None,
        "department": "가족자원경영학과",
        "seed_url": "https://family.sookmyung.ac.kr/family/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "숙명여자대학교",
        "college": None,
        "department": "식품영양학과",
        "seed_url": "https://fn.sookmyung.ac.kr/fn/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "숙명여자대학교",
        "college": None,
        "department": "의류학과",
        "seed_url": "https://cloth.sookmyung.ac.kr/cloth/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "숙명여자대학교",
        "college": None,
        "department": "아동복지학부",
        "seed_url": "https://childwelfare.sookmyung.ac.kr/childwelfare/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "숙명여자대학교",
        "college": "사회과학대학",
        "department": "소비자경제학과",
        "seed_url": "https://conecon.sookmyung.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 동덕여대
    {
        "university": "동덕여자대학교",
        "college": None,
        "department": "패션디자인전공",
        "seed_url": "https://fashion.dongduk.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "동덕여자대학교",
        "college": None,
        "department": "아동학과",
        "seed_url": "https://child.dongduk.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "동덕여자대학교",
        "college": None,
        "department": "식품영양학과",
        "seed_url": "https://nutrition.dongduk.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "동덕여자대학교",
        "college": None,
        "department": "화장품학전공",
        "seed_url": "https://cosmetics.dongduk.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 서울여대
    {
        "university": "서울여자대학교",
        "college": None,
        "department": "아동학과",
        "seed_url": "https://childstudy.swu.ac.kr/",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 덕성여대
    {
        "university": "덕성여자대학교",
        "college": None,
        "department": "아동가족학전공",
        "seed_url": "https://www.duksung.ac.kr/hdfs/main.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "덕성여자대학교",
        "college": None,
        "department": "의상디자인전공",
        "seed_url": "https://www.duksung.ac.kr/fashion/main.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "덕성여자대학교",
        "college": None,
        "department": "식품영양학전공",
        "seed_url": "https://www.duksung.ac.kr/fan/main.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "덕성여자대학교",
        "college": None,
        "department": "실내디자인전공",
        "seed_url": "https://www.duksung.ac.kr/interior/main.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "덕성여자대학교",
        "college": None,
        "department": "텍스타일디자인전공",
        "seed_url": "https://www.duksung.ac.kr/textile/main.do",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 인하대
    {
        "university": "인하대학교",
        "college": None,
        "department": "식품영양학과",
        "seed_url": "https://foodnutri.inha.ac.kr/foodnutri/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "인하대학교",
        "college": None,
        "department": "소비자학과",
        "seed_url": "https://consumer.inha.ac.kr/consumer/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "인하대학교",
        "college": None,
        "department": "아동심리학과",
        "seed_url": "https://child.inha.ac.kr/child/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "인하대학교",
        "college": None,
        "department": "의류디자인학과",
        "seed_url": "https://fashion.inha.ac.kr/fashion/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "인하대학교",
        "college": None,
        "department": "바이오식품공학과",
        "seed_url": "https://foodscience.inha.ac.kr/foodscience/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 단국대
    {
        "university": "단국대학교",
        "college": None,
        "department": "식품영양학과",
        "seed_url": "https://cms.dankook.ac.kr/web/foodnutrition",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "단국대학교",
        "college": None,
        "department": "식품공학과",
        "seed_url": "https://cms.dankook.ac.kr/web/food",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "단국대학교",
        "college": None,
        "department": "식품자원경제학과",
        "seed_url": "https://cms.dankook.ac.kr/web/ere",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "단국대학교",
        "college": None,
        "department": "패션산업디자인전공",
        "seed_url": "https://cms.dankook.ac.kr/web/fashion",
        "max_pages": 120,
        "max_depth": 2,
    },

    # 교원대 / 교대
    {
        "university": "한국교원대학교",
        "college": None,
        "department": "가정교육과",
        "seed_url": "https://www.knue.ac.kr/homeedu/index.do",
        "max_pages": 120,
        "max_depth": 2,
    },
    {
        "university": "서울교육대학교",
        "college": "대학원",
        "department": "초등생활과학교육(석사)",
        "seed_url": "https://grad.snue.ac.kr/grad/cm/cntnts/cntntsView.do?mi=3167&cntntsId=3122",
        "max_pages": 40,
        "max_depth": 1,
    },
    {
        "university": "서울교육대학교",
        "college": "대학원",
        "department": "초등생활과학/컴퓨터교육(박사)",
        "seed_url": "https://grad.snue.ac.kr/grad/cm/cntnts/cntntsView.do?mi=3241&cntntsId=3181",
        "max_pages": 40,
        "max_depth": 1,
    },
    {
        "university": "경인교육대학교",
        "college": None,
        "department": "생활과학교육과",
        "seed_url": "https://edu.ginue.ac.kr/practical/Main.do",
        "max_pages": 120,
        "max_depth": 2,
    },
]

In [99]:
def run_standalone_batch(driver, config_list):
    dfs = []
    for cfg in config_list:
        print(f"[STANDALONE] {cfg['university']} | {cfg['department']}")
        df = crawl_standalone_site(
            driver=driver,
            university=cfg["university"],
            college=cfg["college"],
            department=cfg["department"],
            seed_url=cfg["seed_url"],
            max_pages=cfg["max_pages"],
            max_depth=cfg["max_depth"],
        )
        if not df.empty:
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def run_shared_batch(driver, config_list):
    dfs = []
    for cfg in config_list:
        print(f"[SHARED] {cfg['university']} | {cfg['department']}")

        if cfg["mode"] == "prefix":
            df = crawl_shared_site_by_prefix(
                driver=driver,
                university=cfg["university"],
                college=cfg["college"],
                department=cfg["department"],
                seed_url=cfg["seed_url"],
                allowed_prefixes=cfg["allowed_prefixes"],
                max_pages=cfg["max_pages"],
                max_depth=cfg["max_depth"],
            )
        elif cfg["mode"] == "menu":
            df = crawl_shared_site_by_menu(
                driver=driver,
                university=cfg["university"],
                college=cfg["college"],
                department=cfg["department"],
                seed_url=cfg["seed_url"],
                allowed_menu_nos=cfg["allowed_menu_nos"],
                max_pages=cfg["max_pages"],
                max_depth=cfg["max_depth"],
            )
        else:
            raise ValueError(f"Unknown mode: {cfg['mode']}")

        if not df.empty:
            dfs.append(df)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

## run

In [ ]:
if __name__ == "__main__":
    driver = create_driver(headless=False)

    try:
        df_shared = run_shared_batch(driver, shared_config)
        df_standalone = run_standalone_batch(driver, standalone_config)

        final_df = pd.concat([df_shared, df_standalone], ignore_index=True)
        final_df = final_df.drop_duplicates(subset=["department", "url"]).reset_index(drop=True)

        print("\n[최종 수집 결과]")
        print(final_df["university"].value_counts(dropna=False))
        print(final_df["page_type"].value_counts(dropna=False))

        final_df.to_csv("univ_esg_text_dataset.csv", index=False, encoding="utf-8-sig")

    finally:
        driver.quit()

[SHARED] 연세대학교 | 의류환경학과
[SHARED] 연세대학교 | 식품영양학과
[SHARED] 연세대학교 | 실내건축학과
[SHARED] 연세대학교 | 아동가족학과
[SHARED] 연세대학교 | 통합디자인학과
[SHARED] 경희대학교 | 아동가족학과
[SHARED] 경희대학교 | 주거환경학과
[SHARED] 경희대학교 | 의상학과
[SHARED] 경희대학교 | 식품영양학과
[SHARED] 이화여자대학교 | 섬유패션학부
[SHARED] 서울여자대학교 | 식품영양학과
[SHARED] 서울여자대학교 | 패션산업학과
[SHARED] 서울여자대학교 | 스마트영양관리
[SHARED] 서울여자대학교 | (융합)식품개발실무
[SHARED] 경인교육대학교 | 생활과학교육과
[STANDALONE] 서울대학교 | 생활과학대학
[STANDALONE] 서울대학교 | 식품영양학과
[STANDALONE] 서울대학교 | 의류학과
[ERROR] https://clothing.snu.ac.kr/%ed%95%99%ea%b3%bc%ec%86%8c%ec%8b%9d/?mod=editor&parent_uid=9120 | Alert Text: You do not have permission.
Message: unexpected alert open: {Alert text : You do not have permission.}
  (Session info: chrome=146.0.7680.177)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff64b2f29c5+2ed785]
	chromedriver!GetHandleVerifier [0x7ff64b01a0d0+14e90]
	chromedriver!(No symbol) [0x7ff64ad7db2d]
	chromedriver!(No symbol) [0x7ff64ae2461b]
	chromedriver!(No symbol) [0x7ff64adc9298]
	chromedriver!(No symbol) [0x7ff

## save

In [102]:
data_save_path = r"C:\Users\legen\Desktop\Lab Project\ESG\data"

In [103]:
final_df.to_csv(data_save_path + r"\univ_esg_text_dataset.csv", index=False, encoding="utf-8-sig")

1. 경희대학교 shared 링크 상세 페이지 크롤링 제대로 안됨 -> 따로 잡아서 하기

2. 서울여대의 경우도 shared 링크의 상세 페이지 크롤링 제대로 안됨

In [110]:
final_df.university.value_counts(dropna=False)

university
이화여자대학교    730
고려대학교      509
숙명여자대학교    505
서울대학교      478
인하대학교      397
성균관대학교     378
동덕여자대학교    304
단국대학교      290
건국대학교      237
덕성여자대학교    157
연세대학교      114
한국교원대학교    108
중앙대학교      105
경인교육대학교     96
한양대학교       87
서울여자대학교     22
경희대학교        9
Name: count, dtype: int64

In [112]:
final_df[final_df['university'] == '서울여자대학교'].head(10)

,university,college,department,page_type,url,title,text,selector_used
125,서울여자대학교,자연과학대학,식품영양학과,faculty_index,https://www.swu.ac.kr/www/natusciencej_1.html,서울여자대학교 - 식품영양학과,ENG CHN\n대학소개\n입학정보\n대학/대학원\n대학생활\n발전기금/홍보\n열린...,body
126,서울여자대학교,미래산업융합대학,패션산업학과,faculty_index,https://www.swu.ac.kr/www/futureb_1.html,서울여자대학교 - 패션산업학과,ENG CHN\n대학소개\n입학정보\n대학/대학원\n대학생활\n발전기금/홍보\n열린...,body
127,서울여자대학교,마이크로전공,스마트영양관리,notice,https://www.swu.ac.kr/www/microj_1.html,서울여자대학교 - 스마트영양관리,ENG CHN\n대학소개\n입학정보\n대학/대학원\n대학생활\n발전기금/홍보\n열린...,body
128,서울여자대학교,마이크로전공,스마트영양관리,notice,https://www.swu.ac.kr/www/microj_1.html,서울여자대학교 - 스마트영양관리,ENG CHN\n대학소개\n입학정보\n대학/대학원\n대학생활\n발전기금/홍보\n열린...,body
129,서울여자대학교,마이크로전공,(융합)식품개발실무,notice,https://www.swu.ac.kr/www/microw_1.html,서울여자대학교 - (융합)식품개발실무,ENG CHN\n대학소개\n입학정보\n대학/대학원\n대학생활\n발전기금/홍보\n열린...,body
3526,서울여자대학교,None,아동학과,about,https://childstudy.swu.ac.kr/,서울여자대학교 아동학과,HOME\n학과 소개\n학부\n대학원\n열린광장\n회원가입\nPrevious\nNe...,body
3527,서울여자대학교,None,아동학과,faculty_index,https://childstudy.swu.ac.kr/skin/page/intro03...,서울여자대학교 아동학과,HOME\n학과 소개\n학부\n대학원\n열린광장\n회원가입\n학과 소개\n학과 소개...,body
3528,서울여자대학교,None,아동학과,notice,https://childstudy.swu.ac.kr/bbs/bbs/?bbs_no=8,서울여자대학교 아동학과,HOME\n학과 소개\n학부\n대학원\n열린광장\n회원가입\n열린광장\n열린광장\n...,body
3529,서울여자대학교,None,아동학과,research,https://childstudy.swu.ac.kr/skin/page/graduat...,서울여자대학교 아동학과,HOME\n학과 소개\n학부\n대학원\n열린광장\n회원가입\n대학원\n대학원\n일반...,body
3530,서울여자대학교,None,아동학과,research,https://childstudy.swu.ac.kr/skin/page/college...,서울여자대학교 아동학과,HOME\n학과 소개\n학부\n대학원\n열린광장\n회원가입\n학부\n학부\n교과목정...,body


# khu

In [142]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qs
from collections import deque

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    UnexpectedAlertPresentException,
    NoAlertPresentException,
)

# =========================================================
# 1. 기본 설정
# =========================================================
CURRENT_YEAR = datetime.now().year
NOTICE_YEAR_LIMIT = CURRENT_YEAR - 3
MIN_TEXT_LEN = 120


# =========================================================
# 2. 드라이버
# =========================================================
def create_driver(headless=False):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1600,1200")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--lang=ko-KR")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(30)
    return driver


# =========================================================
# 3. URL / alert 유틸
# =========================================================
def normalize_url(url: str) -> str:
    parsed = urlparse(url)
    return parsed._replace(fragment="").geturl()


def is_same_domain(url: str, base_domain: str) -> bool:
    try:
        return urlparse(url).netloc == base_domain
    except Exception:
        return False


def get_menu_no(url: str):
    try:
        qs = parse_qs(urlparse(url).query)
        return qs.get("menuNo", [None])[0]
    except Exception:
        return None


def should_visit_url(url: str) -> bool:
    url_lower = (url or "").lower()

    blocked_ext = [
        ".jpg", ".jpeg", ".png", ".gif", ".svg", ".pdf",
        ".doc", ".docx", ".xls", ".xlsx", ".zip", ".hwp",
        ".mp4", ".avi", ".mov"
    ]
    if any(url_lower.endswith(ext) for ext in blocked_ext):
        return False

    blocked_keywords = [
        "mailto:", "javascript:", "tel:",
        "filedown.do", "download", "mod=editor", "write", "modify", "delete", "admin"
    ]
    if any(k in url_lower for k in blocked_keywords):
        return False

    return True


def dismiss_alert_if_exists(driver):
    try:
        alert = driver.switch_to.alert
        print(f"[ALERT CLOSED] {alert.text}")
        alert.accept()
        return True
    except NoAlertPresentException:
        return False
    except Exception:
        return False

In [143]:
# =========================================================
# 4. dead / english / forbidden
# =========================================================
def is_dead_page(title: str, text: str) -> bool:
    combined = f"{title}\n{text}".lower()
    patterns = [
        "요청하신 페이지가 존재하지 않습니다",
        "존재하지 않습니다",
        "페이지를 찾을 수 없습니다",
        "404",
        "not found",
    ]
    return any(p in combined for p in patterns)


def is_forbidden_page(title: str, text: str) -> bool:
    combined = f"{title}\n{text}".lower()
    patterns = [
        "you do not have permission",
        "권한이 없습니다",
        "접근 권한이 없습니다",
    ]
    return any(p in combined for p in patterns)


def is_english_page(url: str, title: str = "", text: str = "") -> bool:
    url_lower = (url or "").lower()
    title_lower = (title or "").lower()

    english_url_patterns = ["/eng/", "/english/", "/en/", "lang=en", "locale=en"]
    if any(p in url_lower for p in english_url_patterns):
        return True

    english_title_patterns = [
        "college of", "department of", "faculty",
        "undergraduate", "graduate program", "admission", "curriculum"
    ]
    if any(p in title_lower for p in english_title_patterns):
        return True

    sample = (text or "")[:1200]
    korean_chars = len(re.findall(r"[가-힣]", sample))
    english_words = len(re.findall(r"[A-Za-z]{2,}", sample))

    if korean_chars < 20 and english_words >= 30:
        return True

    return False


# =========================================================
# 5. 노이즈 제거
# =========================================================
NOISE_PATTERNS = [
    "로그인", "KHU Home", "Webmail", "검색창 열기", "검색창 닫기",
    "본문 바로가기", "주메뉴 바로가기", "전체메뉴", "검색", "TOP",
    "교내주요사이트", "개인정보처리방침", "COPYRIGHT © KYUNG HEE UNIVERSITY. ALL RIGHT RESERVED."
]

FOOTER_PATTERNS = [
    "서울캠퍼스 02447 서울특별시 동대문구 경희대로 26",
    "국제캠퍼스 17104 경기도 용인시 기흥구 덕영대로 1732",
    "광릉캠퍼스 12001 경기도 남양주시 진접읍 광릉수목원로 195",
    "COPYRIGHT © KYUNG HEE UNIVERSITY",
]


def clean_text(text: str) -> str:
    cleaned = text
    for p in NOISE_PATTERNS:
        cleaned = cleaned.replace(p, "")
    cleaned = re.sub(r"\n{2,}", "\n", cleaned)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    return cleaned.strip()


# =========================================================
# 6. 날짜 / page_type
# =========================================================
def extract_year_from_text(text: str):
    if not text:
        return None
    match = re.search(r"(20\d{2}|19\d{2})", text)
    if match:
        year = int(match.group(1))
        if 1900 <= year <= CURRENT_YEAR + 1:
            return year
    return None


def is_recent_notice(title: str, text: str, url: str) -> bool:
    combined = f"{title}\n{text}\n{url}"
    year = extract_year_from_text(combined)
    if year is None:
        return True
    return year >= NOTICE_YEAR_LIMIT


def is_notice_like(title: str, url: str, text: str = "") -> bool:
    combined = f"{title}\n{url}\n{text}".lower()
    keywords = [
        "공지", "notice", "게시판", "board", "news", "자료실",
        "학과소식", "학부공지", "대학원공지", "faq", "q&a"
    ]
    return any(k in combined for k in keywords)


def is_post_like_page(title: str, url: str, text: str = "") -> bool:
    combined = f"{title}\n{url}\n{text}".lower()
    if re.search(r"(20\d{2})[.\-/](0?[1-9]|1[0-2])[.\-/](0?[1-9]|[12]\d|3[01])", combined):
        return True
    post_keywords = ["게시판", "board", "공지", "공지사항", "자료실", "faq", "q&a"]
    return any(k in combined for k in post_keywords)


def classify_page_type(title: str, url: str, text: str = "") -> str:
    combined = f"{title}\n{url}\n{text}".lower()

    if is_notice_like(title, url, text):
        return "notice"
    if is_post_like_page(title, url, text):
        return "board"

    rules = [
        ("faculty_index", ["교수소개", "교수진", "전임교수", "명예교수", "전임교원", "명예교원", "faculty"]),
        ("research", ["연구", "연구실", "실험실", "논문", "프로젝트", "lab", "research"]),
        ("curriculum", ["교육과정", "교과과정", "curriculum"]),
        ("course", ["교과목", "과목소개", "course"]),
        ("requirements", ["내규", "시행세칙", "졸업요건", "이수기준", "학사규정"]),
        ("program", ["연계", "복수전공", "협동", "융합", "프로그램"]),
        ("career", ["진로", "취업", "자격증"]),
        ("student", ["학생", "졸업생", "인터뷰"]),
        ("activity", ["심포지엄", "세미나", "행사", "학과활동", "학생활동", "졸업작품"]),
        ("about", ["학과소개", "소개", "비전", "교육목표", "인사말", "대학소개", "연혁 및 현황", "위치 및 연락처"]),
    ]

    for page_type, patterns in rules:
        if any(p.lower() in combined for p in patterns):
            return page_type

    return "etc"

In [144]:
# =========================================================
# 7. 교수 컨텍스트 / 더보기
# =========================================================
parent_context_patterns = [
    "교수", "교수진", "교수소개", "교수 소개",
    "전임교수", "명예교수", "특임교수", "겸임교수",
    "전임교원", "명예교원", "겸임교원",
    "식품학분야", "영양학분야",
    "es 교수", "is 교수", "es교수", "is교수", "ES/IS 교수",
    "faculty"
]


def is_khu_faculty_context(title: str, text: str) -> bool:
    combined = f"{title}\n{text}".lower()
    return any(p.lower() in combined for p in parent_context_patterns)


def expand_khu_more_buttons(driver):
    """
    식품영양학과 교수소개처럼 같은 페이지에 '더보기'가 있는 경우를 보수적으로 클릭
    """
    xpaths = [
        "//button[contains(normalize-space(.), '더보기')]",
        "//a[contains(normalize-space(.), '더보기')]",
        "//button[contains(translate(normalize-space(.), 'MORE', 'more'), 'more')]",
        "//a[contains(translate(normalize-space(.), 'MORE', 'more'), 'more')]",
    ]

    negative_keywords = ["print", "프린트", "download", "자료다운", "첨부", "pdf", "hwp", "xlsx"]
    clicked = 0
    seen = set()

    for xp in xpaths:
        try:
            elems = driver.find_elements(By.XPATH, xp)
        except Exception:
            elems = []

        for elem in elems:
            try:
                text = (elem.text or "").strip().lower()
                href = (elem.get_attribute("href") or "").lower()
                onclick = (elem.get_attribute("onclick") or "").lower()
                meta = " ".join([text, href, onclick])

                if any(k in meta for k in negative_keywords):
                    continue

                key = driver.execute_script("return arguments[0].outerHTML;", elem)[:300]
                if key in seen:
                    continue
                seen.add(key)

                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", elem)
                time.sleep(0.2)
                driver.execute_script("arguments[0].click();", elem)
                time.sleep(0.4)
                clicked += 1
            except Exception:
                continue

    return clicked


# =========================================================
# 8. 경희대 본문 추출
#    - article / .cont 우선
#    - .txt 푸터 강하게 배제
# =========================================================
def score_khu_block(text: str, selector: str) -> int:
    if not text:
        return -99999

    score = 0
    length = len(text)
    score += min(length, 4000)

    selector_bonus = {
        "article": 3000,
        ".cont": 2500,
        "main": 2000,
        "#content": 1800,
        ".content": 1600,
        ".contents": 1500,
        ".sub_content": 1500,
        ".board_view": 1400,
        ".view": 1200,
        "body": 300,
        ".txt": -6000,
    }
    score += selector_bonus.get(selector, 0)

    footer_hits = sum(1 for p in FOOTER_PATTERNS if p in text)
    score -= footer_hits * 4000

    menu_hits = sum(1 for p in ["로그인", "Webmail", "전체메뉴", "검색", "교내주요사이트"] if p in text)
    score -= menu_hits * 500

    if "COPYRIGHT" in text.upper():
        score -= 2500

    korean_chars = len(re.findall(r"[가-힣]", text))
    if korean_chars >= 80:
        score += 500

    return score


def extract_khu_main_text(driver):
    candidate_selectors = [
        "article",
        ".cont",
        "main",
        "#content",
        ".content",
        ".contents",
        ".sub_content",
        ".board_view",
        ".view",
        "body",
        ".txt",
    ]

    candidates = []

    for sel in candidate_selectors:
        try:
            elems = driver.find_elements(By.CSS_SELECTOR, sel)
        except Exception:
            elems = []

        for elem in elems[:3]:
            try:
                raw = elem.text.strip()
                if not raw:
                    continue
                cleaned = clean_text(raw)
                score = score_khu_block(cleaned, sel)
                candidates.append((score, sel, cleaned))
            except Exception:
                continue

    if not candidates:
        return None, ""

    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_selector, best_text = candidates[0]

    if len(best_text) < MIN_TEXT_LEN:
        return best_selector, ""

    return best_selector, best_text


# =========================================================
# 9. 링크 수집
# =========================================================
def collect_khu_internal_links(driver, current_url: str, base_domain: str, allowed_menu_nos: set):
    soup = BeautifulSoup(driver.page_source, "html.parser")
    found = set()

    for a in soup.find_all("a", href=True):
        href = normalize_url(urljoin(current_url, a["href"]))
        if not should_visit_url(href):
            continue
        if not is_same_domain(href, base_domain):
            continue

        menu_no = get_menu_no(href)
        if menu_no and menu_no in allowed_menu_nos:
            found.add(href)

    return list(found)


# =========================================================
# 10. 경희대 메뉴 맵
#     - 현재 사이트 구조 확인 결과 기준 수동 고정
# =========================================================
def contents_url(menu_no: int) -> str:
    return f"https://che.khu.ac.kr/che/user/contents/view.do?menuNo={menu_no}"

def bbs40_url(menu_no: int) -> str:
    return f"https://che.khu.ac.kr/che/user/bbs/BMSR00040/list.do?menuNo={menu_no}"

def bbs45_url(menu_no: int) -> str:
    return f"https://che.khu.ac.kr/che/user/bbs/BMSR00045/list.do?menuNo={menu_no}"

def bbs47_url(menu_no: int) -> str:
    return f"https://che.khu.ac.kr/che/user/bbs/BMSR00047/list.do?menuNo={menu_no}"

def bbs48_url(menu_no: int) -> str:
    return f"https://che.khu.ac.kr/che/user/bbs/BMSR00048/list.do?menuNo={menu_no}"


KHU_MENU_MAP = {
    "생활과학대학": [
        contents_url(12700171),  # 대학소개
        contents_url(12700008),  # 연혁 및 현황
        contents_url(12700009),  # 교수 및 직원소개
        bbs40_url(12700018),     # 생활과학연구소
        bbs40_url(12700182),     # 공지사항
        bbs40_url(12700019),     # 취업·홍보
        contents_url(12700020),  # 지침·서식
        bbs48_url(12700021),     # 장소·기자재 대여 안내
        contents_url(12700025),  # 생활과학대학 소식지
        contents_url(12700027),  # Brochure
        contents_url(12700024),  # 발전기금 안내 / 위치 및 연락처 계열
        bbs45_url(12700158),     # 갤러리
        "https://che.khu.ac.kr/che/user/main/view.do",  # 메인
    ],
    "아동가족학과": [
        contents_url(12700011),  # 학과소개
        bbs47_url(12700037),     # 교수소개
        contents_url(12700031),  # 교육과정
        contents_url(12700228),  # 내규·시행세칙
        contents_url(12700039),  # 대학원
        contents_url(12700034),  # 학과특성화
        bbs45_url(12700227),     # 학과활동
        bbs40_url(12700036),     # 공지사항(아가)
        contents_url(12700042),  # English
    ],
    "주거환경학과": [
        contents_url(12700012),  # 학과소개
        bbs47_url(12700051),     # 교수소개
        contents_url(12700044),  # 교육과정
        contents_url(12700188),  # 내규·시행세칙
        contents_url(12700156),  # 대학원
        bbs45_url(12700095),     # 교육특성화
        bbs45_url(12700160),     # 학과활동
        bbs40_url(12700049),     # 공지사항(주거)
        contents_url(12700050),  # English
    ],
    "의상학과": [
        contents_url(12700013),  # 학과소개
        bbs47_url(12700057),     # 교수소개
        contents_url(12700058),  # 교육과정
        contents_url(12700230),  # 내규·시행세칙
        contents_url(12700066),  # 대학원
        contents_url(12700164),  # 학과특성화
        contents_url(12700072),  # 학생활동
        contents_url(12700197),  # 졸업작품
        bbs40_url(12700064),     # 공지사항(의상)
    ],
    "식품영양학과": [
        contents_url(12700014),  # 학과소개
        bbs47_url(12700081),     # 교수소개 > 식품학분야
        bbs47_url(12700082),     # 교수소개 > 영양학분야
        bbs47_url(12700083),     # 교수소개 > ES/IS 교수
        bbs47_url(12700084),     # 교수소개 > 명예교수
        contents_url(12700074),  # 교육과정
        contents_url(12700231),  # 내규·시행세칙
        contents_url(12700085),  # 대학원
        contents_url(12700077),  # 실험실 소개
        contents_url(12700078),  # 학생 및 학과활동
        contents_url(12700169),  # 위탁사업
        bbs40_url(12700079),     # 공지사항(식영)
        contents_url(12700080),  # English
    ],
}

In [145]:
# =========================================================
# 11. 학과/대학 1개 크롤링
# =========================================================
def crawl_khu_one_entity(driver, university: str, college: str, department: str,
                         start_urls: list, max_pages: int = 150, max_depth: int = 2):
    base_domain = "che.khu.ac.kr"
    allowed_menu_nos = {mn for mn in [get_menu_no(u) for u in start_urls] if mn is not None}
    queue = deque([(normalize_url(u), 0) for u in start_urls])
    visited = set()
    rows = []

    while queue and len(visited) < max_pages:
        current_url, depth = queue.popleft()
        current_url = normalize_url(current_url)

        if current_url in visited:
            continue
        if depth > max_depth:
            continue

        try:
            try:
                driver.get(current_url)
                time.sleep(1.2)
            except UnexpectedAlertPresentException:
                dismiss_alert_if_exists(driver)
                print(f"[SKIP ALERT URL] {current_url}")
                continue

            visited.add(current_url)

            soup = BeautifulSoup(driver.page_source, "html.parser")
            title = soup.title.get_text(" ", strip=True) if soup.title else ""
            full_text = soup.get_text("\n", strip=True)

            if is_dead_page(title, full_text):
                continue
            if is_forbidden_page(title, full_text):
                continue

            # 교수 페이지면 same-page 더보기 시도
            if is_khu_faculty_context(title, full_text[:1000]):
                expand_khu_more_buttons(driver)
                time.sleep(0.3)

            selector_used, text = extract_khu_main_text(driver)
            if is_english_page(current_url, title, text or full_text):
                continue

            if text and len(text) >= MIN_TEXT_LEN:
                page_type = classify_page_type(title, current_url, text)

                if page_type in ["notice", "board"]:
                    if is_recent_notice(title, text, current_url):
                        rows.append({
                            "university": university,
                            "college": college,
                            "department": department,
                            "page_type": page_type,
                            "url": current_url,
                            "title": title,
                            "text": text,
                            "selector_used": selector_used,
                        })
                else:
                    rows.append({
                        "university": university,
                        "college": college,
                        "department": department,
                        "page_type": page_type,
                        "url": current_url,
                        "title": title,
                        "text": text,
                        "selector_used": selector_used,
                    })

            next_links = collect_khu_internal_links(driver, current_url, base_domain, allowed_menu_nos)
            for link in next_links:
                if link not in visited:
                    queue.append((link, depth + 1))

        except Exception as e:
            dismiss_alert_if_exists(driver)
            print(f"[KHU ERROR] {current_url} | {e}")

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["department", "url"]).reset_index(drop=True)

    print(f"[KHU DONE] {department} | {0 if df.empty else len(df)} rows")
    return df


# =========================================================
# 12. 경희대 전체 실행
# =========================================================
def run_khu_crawl(headless=False):
    driver = create_driver(headless=headless)

    try:
        all_df = []

        order = [
            "생활과학대학",
            "아동가족학과",
            "주거환경학과",
            "의상학과",
            "식품영양학과",
        ]

        for dept in order:
            df_dept = crawl_khu_one_entity(
                driver=driver,
                university="경희대학교",
                college="생활과학대학",
                department=dept,
                start_urls=KHU_MENU_MAP[dept],
                max_pages=200 if dept == "식품영양학과" else 120,
                max_depth=2 if dept == "생활과학대학" else 2,
            )
            if not df_dept.empty:
                all_df.append(df_dept)

        if not all_df:
            final_df = pd.DataFrame(columns=[
                "university", "college", "department",
                "page_type", "url", "title", "text", "selector_used"
            ])
        else:
            final_df = pd.concat(all_df, ignore_index=True)
            final_df = final_df.drop_duplicates(subset=["department", "url"]).reset_index(drop=True)

        print("\n[경희대 전체 결과]")
        print(final_df["department"].value_counts(dropna=False))
        print(final_df["page_type"].value_counts(dropna=False))
        print(final_df.shape)

        return final_df

    finally:
        driver.quit()

## run

In [146]:
# 실행
if __name__ == "__main__":
    df_khu = run_khu_crawl(headless=False)
    df_khu.to_csv(data_save_path +r"\df_khu.csv", index=False, encoding="utf-8-sig")

[KHU DONE] 생활과학대학 | 14 rows
[KHU DONE] 아동가족학과 | 7 rows
[KHU DONE] 주거환경학과 | 7 rows
[KHU DONE] 의상학과 | 7 rows
[KHU DONE] 식품영양학과 | 11 rows

[경희대 전체 결과]
department
생활과학대학    14
식품영양학과    11
아동가족학과     7
주거환경학과     7
의상학과       7
Name: count, dtype: int64
page_type
research         20
notice           10
faculty_index     9
board             5
about             1
student           1
Name: count, dtype: int64
(46, 8)


## save

In [122]:
final_df = final_df.drop(final_df[final_df['university'] == '경희대학교'].index).reset_index(drop=True)

In [151]:
final_df = pd.concat([final_df, df_khu], ignore_index=True)
final_df = final_df.drop_duplicates(subset=["department", "url"]).reset_index(drop=True)

In [152]:
final_df.to_csv(data_save_path +r"\univ_esg_text_dataset.csv", index=False, encoding="utf-8-sig")

In [153]:
final_df.head()

,university,college,department,page_type,url,title,text,selector_used
0,연세대학교,생활과학대학,의류환경학과,research,https://che.yonsei.ac.kr/che/departments/cloth...,생활과학대학 | 학과소개 | 의류환경학과 | 학과소개,의류환경학과는 의류산업발전에 기여하고 의류학을 발전시키고 패션산업을 이끌어나갈 인재...,.content
1,연세대학교,생활과학대학,의류환경학과,notice,https://che.yonsei.ac.kr/che/departments/cloth...,의류환경학과 교수진 게시판목록 | 생활과학대학,현교수진\n명예•퇴임교수\n전체\n고은주\nProfessor\nejko@yonsei...,.content
2,연세대학교,생활과학대학,의류환경학과,research,https://che.yonsei.ac.kr/che/departments/cloth...,생활과학대학 | 학과소개 | 의류환경학과 | 졸업 후 전망과 진로,"의류환경학과는 의류환경의 제반현상을 분석하고 인식할 수 있는 능력을 형성하고, 의류...",.content
3,연세대학교,생활과학대학,의류환경학과,research,https://che.yonsei.ac.kr/che/departments/cloth...,생활과학대학 | 학과소개 | 의류환경학과 | 대학원 교과목,의류환경학과의 대학원 교과과정은 의류산업을 구성하는 여러 요소와 인간과 의류의 상호...,.content
4,연세대학교,생활과학대학,의류환경학과,notice,https://che.yonsei.ac.kr/che/departments/cloth...,의류환경학과 게시판 게시판목록 | 생활과학대학,전체\n번호 제목 첨부 작성자 등록일\n공지 [대학원] 2026-1학기 종합시험 안...,.content


# check

In [ ]:
import os
import re
import hashlib
import numpy as np
import pandas as pd

# 복사본 사용
df = final_df.copy()

# =========================
# 1. 기본 정리
# =========================
text_cols = ["university", "college", "department", "page_type", "url", "title", "text", "selector_used"]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

def normalize_text_basic(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

df["text_raw"] = df["text"]
df["text"] = df["text"].apply(normalize_text_basic)

# page_id 부여
df = df.reset_index(drop=True)
df["page_id"] = df.index + 1

# 길이 지표
df["char_len"] = df["text"].str.len()
df["line_count"] = df["text"].str.count("\n") + 1
df["word_count_kr_rough"] = df["text"].str.split().str.len()

# 해시(중복 탐지용)
def md5_hash(x: str) -> str:
    return hashlib.md5(x.encode("utf-8", errors="ignore")).hexdigest()

df["text_hash"] = df["text"].apply(md5_hash)

In [ ]:
# =========================
# 2. 분석용 그룹 분리
# =========================
# 핵심 분석 corpus / 보조 / 검토 필요
core_page_types = {
    "about", "research", "faculty_index", "career",
    "course", "curriculum", "program", "requirements",
    "student", "activity"
}
noise_page_types = {"notice", "board"}

def classify_analysis_group(page_type: str) -> str:
    if page_type in core_page_types:
        return "core"
    elif page_type in noise_page_types:
        return "noise"
    else:
        return "review"

df["analysis_group"] = df["page_type"].apply(classify_analysis_group)

In [ ]:
# =========================
# 3. 품질 점검 테이블 생성
# =========================
# 3-1. 전체 요약
qc_summary = pd.DataFrame({
    "metric": [
        "n_rows",
        "n_university",
        "n_college",
        "n_department",
        "n_unique_url",
        "n_duplicate_url_rows",
        "n_duplicate_text_rows",
        "n_empty_text",
        "char_len_mean",
        "char_len_median",
        "char_len_max",
        "core_rows",
        "noise_rows",
        "review_rows"
    ],
    "value": [
        len(df),
        df["university"].nunique(),
        df["college"].nunique(),
        df["department"].nunique(),
        df["url"].nunique(),
        int(df.duplicated(subset=["url"]).sum()),
        int(df.duplicated(subset=["text_hash"]).sum()),
        int((df["text"] == "").sum()),
        round(df["char_len"].mean(), 2),
        round(df["char_len"].median(), 2),
        int(df["char_len"].max()),
        int((df["analysis_group"] == "core").sum()),
        int((df["analysis_group"] == "noise").sum()),
        int((df["analysis_group"] == "review").sum())
    ]
})

# 3-2. page_type 분포
page_type_table = (
    df.groupby("page_type", dropna=False)
      .agg(
          n_pages=("page_id", "count"),
          mean_chars=("char_len", "mean"),
          median_chars=("char_len", "median"),
          min_chars=("char_len", "min"),
          max_chars=("char_len", "max")
      )
      .sort_values("n_pages", ascending=False)
      .reset_index()
)

# 3-3. 대학별 page_type 분포
uni_page_type_table = pd.crosstab(df["university"], df["page_type"])

# 3-4. 대학별 요약
uni_summary = (
    df.groupby("university")
      .agg(
          n_pages=("page_id", "count"),
          n_departments=("department", "nunique"),
          mean_chars=("char_len", "mean"),
          median_chars=("char_len", "median"),
          notice_ratio=("page_type", lambda s: np.mean(s.isin(["notice", "board"]))),
          core_ratio=("analysis_group", lambda s: np.mean(s == "core")),
          review_ratio=("analysis_group", lambda s: np.mean(s == "review"))
      )
      .sort_values(["n_pages", "notice_ratio"], ascending=[False, False])
      .reset_index()
)

# 3-5. 중복 의심
dup_url = df[df.duplicated(subset=["url"], keep=False)].sort_values(["url", "page_id"])
dup_text = df[df.duplicated(subset=["text_hash"], keep=False)].sort_values(["text_hash", "page_id"])

# 3-6. 너무 짧거나 긴 페이지
# 기준은 예비 점검용이므로 보수적으로 둠
short_pages = df[df["char_len"] < 150].sort_values("char_len")
long_pages = df[df["char_len"] > 5000].sort_values("char_len", ascending=False)

# 3-7. 검토 우선 페이지
# - review 그룹
# - 너무 짧은 페이지
# - 너무 긴 페이지
# - selector_used 이상 여부
review_candidates = df[
    (df["analysis_group"] == "review") |
    (df["char_len"] < 150) |
    (df["char_len"] > 5000)
].copy()

# 3-8. 대표 샘플 추출
sample_per_type = (
    df.groupby("page_type", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 3), random_state=42))
      .sort_values(["page_type", "university", "department"])
      .reset_index(drop=True)
)

C:\Users\legen\AppData\Local\Temp\ipykernel_25900\3243903218.py:96: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 3), random_state=42))


In [ ]:
# =========================
# 4. 저장
# =========================
qc_dir = os.path.join(data_save_path, "quality_check")
os.makedirs(qc_dir, exist_ok=True)

qc_summary.to_csv(os.path.join(qc_dir, "qc_summary.csv"), index=False, encoding="utf-8-sig")
page_type_table.to_csv(os.path.join(qc_dir, "page_type_table.csv"), index=False, encoding="utf-8-sig")
uni_page_type_table.to_csv(os.path.join(qc_dir, "uni_page_type_table.csv"), encoding="utf-8-sig")
uni_summary.to_csv(os.path.join(qc_dir, "uni_summary.csv"), index=False, encoding="utf-8-sig")
dup_url.to_csv(os.path.join(qc_dir, "duplicate_url_rows.csv"), index=False, encoding="utf-8-sig")
dup_text.to_csv(os.path.join(qc_dir, "duplicate_text_rows.csv"), index=False, encoding="utf-8-sig")
short_pages.to_csv(os.path.join(qc_dir, "short_pages_under_150chars.csv"), index=False, encoding="utf-8-sig")
long_pages.to_csv(os.path.join(qc_dir, "long_pages_over_5000chars.csv"), index=False, encoding="utf-8-sig")
review_candidates.to_csv(os.path.join(qc_dir, "review_candidates.csv"), index=False, encoding="utf-8-sig")
sample_per_type.to_csv(os.path.join(qc_dir, "sample_per_page_type.csv"), index=False, encoding="utf-8-sig")

print("품질 점검 파일 저장 완료:", qc_dir)
print("\n[qc_summary]")
print(qc_summary)
print("\n[page_type_table 상위]")
print(page_type_table.head(15))
print("\n[uni_summary 상위]")
print(uni_summary.head(15))

품질 점검 파일 저장 완료: C:\Users\legen\Desktop\Lab Project\ESG\data\quality_check

[qc_summary]
                   metric     value
0                  n_rows   4563.00
1            n_university     17.00
2               n_college     21.00
3            n_department     51.00
4            n_unique_url   4372.00
5    n_duplicate_url_rows    191.00
6   n_duplicate_text_rows    871.00
7            n_empty_text      0.00
8           char_len_mean   1059.23
9         char_len_median    719.00
10           char_len_max  29803.00
11              core_rows   1646.00
12             noise_rows   2607.00
13            review_rows    310.00

[page_type_table 상위]
        page_type  n_pages   mean_chars  median_chars  min_chars  max_chars
0          notice     2602  1101.492698         773.0         55      29803
1        research      777  1454.438867         861.0         59      26084
2             etc      310   352.322581         168.5         51       2452
3   faculty_index      204  1517.176471       